In [2]:
import leafmap
import solara
from urllib.parse import quote
from localtileserver import TileClient
import os
from pathlib import Path
import pandas as pd
import geopandas as gpd
from shapely import Polygon
import rioxarray as rxr
import ipywidgets as widgets
import requests
from urllib.parse import quote
import math
import yaml
import duckdb
from contextlib import contextmanager
import time
import logging
import sys

In [7]:

with open('/home/jovyan/solara-labeler/src/settings.yml', 'r') as file:
    settings = yaml.safe_load(file)

data_dir = Path(settings['data_dir'])
years = settings['years']
pre_render = settings['pre_render']
host_base_port = settings['tileserver']['host_base_port']
container_base_port = settings['tileserver']['container_base_port']
preload_chips = settings['preload_chips']
chip_buffer_size = settings['chip_buffer_size']
show_buffer = settings['show_buffer']
host_ip = settings['host_ip']
db_path = data_dir / 'chip_tracker.duckdb'


@contextmanager
def connect_to_db():
    con = None
    while con is None:
        try:
            con=duckdb.connect(str(db_path))
        except duckdb.IOException:
            time.sleep(0.1)
    try:
        yield con
    finally:
        con.close()


In [4]:

zoom = solara.reactive(settings['map']['zoom'])
center = solara.reactive(settings['map']['center'])
current_chip = solara.reactive(None)
current_chip_start_time = solara.reactive(None)
previous_chip = solara.reactive(None)
chip_buffer = solara.reactive(None)
current_year_index = solara.reactive(0)
current_user = solara.reactive("")
success_visible = solara.reactive(False)
error_visible = solara.reactive(False)
success_message = solara.reactive("")
error_message = solara.reactive("")
estimated_time_remaining = solara.reactive(0)


In [15]:
with connect_to_db() as con:
    unique_values = con.execute("SELECT DISTINCT user FROM chip_tracker").fetchall()
    print(unique_values)

[('Gbooker',), ('ATyagi',), ('Solana',), ('ZBaranowski',), ('Glass/Potentially a solar panel',), ('nattia',), ('Annan Shrestha',), ('Potentially a solar panel',), ('Sarala Adhikari',), ('SANJIDA',), ('Denys',), ('solana',), ('',), ('sanjida',), ('jreitinger',), ('pri',)]


In [20]:
current_user.set("Denys")

In [49]:
with connect_to_db() as con:
    # Get all completed chips for this user with start and end times
    user_completed_chips = con.execute(
        "SELECT start_time, end_time FROM chip_tracker WHERE status = ? AND user = ?",
        ('labeled', current_user.value)
    ).fetchall()
    pending_chips = con.execute(
        "SELECT COUNT(*) FROM chip_tracker WHERE status = ?",
        ['pending']
    ).fetchone()[0]

total = timedelta(0)
count = len(user_completed_chips)

for chip in user_completed_chips:
    start_time = datetime.fromisoformat(chip[0])
    end_time = datetime.fromisoformat(chip[1])
    total += end_time - start_time

mean_timedelta = total / count if count > 0 else timedelta(0)
estimated_time_remaining = pending_chips * mean_timedelta


In [50]:
estimated_time_remaining

datetime.timedelta(days=7, seconds=59769, microseconds=949050)

In [44]:
from datetime import datetime, timedelta


In [ ]:

    
    # Calculate average time to complete (in seconds or minutes depending on your timestamp format)
    if user_completed_chips:
        total_time = 0
        for chip in user_completed_chips:
            start_time = chip[0]  # or chip['start_time'] if using row factory
            end_time = chip[1]    # or chip['end_time'] if using row factory
            
            # Calculate duration (assumes datetime objects)
            duration = (end_time - start_time).total_seconds()
            total_time += duration
        
        avg_time_per_chip = total_time / len(user_completed_chips)
    else:
        # No completed chips yet, can't estimate
        return None
    
    # Get count of pending chips for this user
    pending_count = con.execute(
        "SELECT COUNT(*) FROM chip_tracker WHERE status = ?",
        ('pending')
    ).fetchone()[0]
    
    # Estimate total time remaining
    estimated_time_remaining = avg_time_per_chip * pending_count
    
    return {
        'completed_chips': len(user_completed_chips),
        'avg_time_per_chip_seconds': avg_time_per_chip,
        'avg_time_per_chip_minutes': avg_time_per_chip / 60,
        'pending_chips': pending_count,
        'estimated_remaining_seconds': estimated_time_remaining,
        'estimated_remaining_minutes': estimated_time_remaining / 60,
        'estimated_remaining_hours': estimated_time_remaining / 3600
    }
    